# Notebook 53 — Prompt Registry

Demonstrates `multigen.prompt_registry`: template compilation with conditionals
and loops, versioned registry, prompt inheritance, A/B testing, access control,
review workflow, and the `PromptManager` facade.  No external APIs needed.

In [ ]:
import sys, importlib.util
sys.path.insert(0, '../sdk')

def load(name):
    spec = importlib.util.spec_from_file_location(
        f'multigen.{name}', f'../sdk/multigen/{name}.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[f'multigen.{name}'] = mod
    spec.loader.exec_module(mod)
    return mod

pr = load('prompt_registry')
print('prompt_registry loaded OK')

## 1. PromptTemplate — variable substitution, conditional block, loop

In [ ]:
# Simple variable substitution
tpl = pr.PromptTemplate('Hello {name}! Your score is {score}.')
print('Simple render:', tpl.render(name='Alice', score=95))

# Conditional block — {%if title%}...{%endif%}
tpl_cond = pr.PromptTemplate(
    'Dear {%if title%}{title} {%endif%}{name}, please review the report.'
)
print('With title   :', tpl_cond.render(name='Smith', title='Dr.'))
print('Without title:', tpl_cond.render(name='Smith', title=''))

# Loop block — {%for item in list%}...{%endfor%}
tpl_loop = pr.PromptTemplate(
    'Key findings:\n{%for item in findings%}- {item}\n{%endfor%}'
)
rendered_loop = tpl_loop.render(
    findings=['Revenue up 14%', 'Margins improved', 'Cloud growth +22%']
)
print('\nLoop render:')
print(rendered_loop)

## 2. PromptRegistry — create v1 and v2, publish v2, get_live returns v2

In [ ]:
registry = pr.PromptRegistry()

v1 = registry.create('qa', 'Answer this question: {question}', author='alice')
v2 = registry.create('qa', 'You are an expert. Answer precisely: {question}', author='alice')

print('All versions for "qa":')
for pv in registry.versions('qa'):
    print(f'  version={pv.version}  status={pv.status}  author={pv.author}')

# Publish v2 only
registry.publish('qa', '2.0')

print('\nAfter publish v2:')
for pv in registry.versions('qa'):
    print(f'  version={pv.version}  status={pv.status}')

live_tpl = registry.get_live('qa')
print('\nget_live("qa") template:', live_tpl.template)
print('Compiled with question:', registry.compile('qa', question='What is ROI?'))

## 3. PromptInheritance — base template with {%block persona%}, override for finance agent

In [ ]:
base_template = (
    'System: {%block persona%}You are a helpful assistant.{%endblock%}\n'
    'Task: {%block task%}Answer the user query.{%endblock%}\n'
    'Question: {question}'
)

# Default build — uses block defaults
default_built = pr.PromptInheritance(base_template).build()
default_tpl = pr.PromptTemplate(default_built)
print('Default agent prompt:')
print(default_tpl.render(question='What is cash flow?'))

# Finance agent — override persona and task blocks
finance_built = (
    pr.PromptInheritance(base_template)
    .override('persona', 'You are a senior financial analyst with CFA certification.')
    .override('task', 'Provide a precise, data-driven answer with relevant metrics.')
    .build()
)
finance_tpl = pr.PromptTemplate(finance_built)
print('\nFinance agent prompt:')
print(finance_tpl.render(question='What is cash flow?'))

## 4. PromptABTest — 50 rounds, simulated scores, result() showing winner

In [ ]:
import random
random.seed(42)

ab = pr.PromptABTest('v1.0', 'v2.0', split=0.5)

# v1.0 scores cluster around 0.70, v2.0 around 0.80
for _ in range(50):
    variant = ab.select()
    if variant == 'v1.0':
        score = random.gauss(0.70, 0.05)
    else:
        score = random.gauss(0.80, 0.05)
    ab.record(variant, score)

result = ab.result()
print(f'Variant A (v1.0): n={result.n_a}  avg_score={result.score_a:.4f}')
print(f'Variant B (v2.0): n={result.n_b}  avg_score={result.score_b:.4f}')
print(f'Winner: {result.winner}')

## 5. PromptAccessControl — alice=admin can publish; bob=writer raises PermissionDeniedError

In [ ]:
acl = pr.PromptAccessControl()
acl.grant('alice', 'admin')
acl.grant('bob',   'writer')

# alice can publish
print('alice can read  :', acl.check('alice', 'read'))
print('alice can create:', acl.check('alice', 'create'))
print('alice can publish:', acl.check('alice', 'publish'))

# bob can create but cannot publish
print('\nbob can create:', acl.check('bob', 'create'))
try:
    acl.check('bob', 'publish')
except pr.PermissionDeniedError as exc:
    print('bob publish → PermissionDeniedError:', exc)

## 6. PromptReviewWorkflow — submit v2 for review, approve, verify live

In [ ]:
reg2 = pr.PromptRegistry()
reg2.create('qa', 'Answer the question: {question}', author='alice')  # v1.0
reg2.create('qa', 'Provide a concise expert answer to: {question}',    author='alice')  # v2.0

workflow = pr.PromptReviewWorkflow(reg2)

# Submit v2.0 for review
review_req = workflow.submit('qa', '2.0', submitted_by='alice')
print('Review request:', review_req.id)
print('Status after submit :', review_req.status)
print('Prompt version status:', reg2.get('qa', '2.0').status)

# Pending reviews
print('Pending reviews:', len(workflow.pending()))

# Approve with comments
approved = workflow.approve(review_req.id, reviewer='bob', comments='Looks good — approved.')
print('\nStatus after approve :', approved.status)
print('Reviewer             :', approved.reviewer)
print('Comments             :', approved.comments)
print('Prompt v2.0 status   :', reg2.get('qa', '2.0').status)

# Verify live
live = reg2.get_live('qa')
print('Live template        :', live.template)

## 7. PromptManager facade — create + publish + compile + ab_test all together

In [ ]:
mgr = pr.PromptManager()
mgr.acl.grant('alice', 'admin')
mgr.acl.grant('bob',   'writer')

# Create two versions
v1 = mgr.create('summarise', 'Summarise the following text: {text}', author='alice')
v2 = mgr.create('summarise', 'Provide a concise 3-sentence summary of: {text}', author='alice')

# Publish v2
mgr.publish('summarise', '2.0')

# Compile
compiled = mgr.compile('summarise', text='The annual report shows strong performance...')
print('Compiled prompt:', compiled)

# A/B test
import random
random.seed(99)
ab = mgr.ab_test('summarise', '1.0', '2.0')
for _ in range(20):
    variant = ab.select()
    ab.record(variant, score=random.uniform(0.6, 0.9))

result = ab.result()
print(f'\nA/B result — winner: {result.winner}')
print(f'  v1.0 avg={result.score_a:.3f} (n={result.n_a})')
print(f'  v2.0 avg={result.score_b:.3f} (n={result.n_b})')
print('All prompts in registry:', mgr.registry.list_prompts())